<div class="freebsdLab-IntroCard">
  <div class="freebsdLab-IntroBrand">2. Lab Demo</div>
  <h1 class="freebsdLab-IntroTitle">FreeBSD Laboratory Demonstration</h1>
  <p class="freebsdLab-IntroTagline">Isolation Verification, Network Containment &amp; Runtime Resilience</p>
  <p class="freebsdLab-IntroCopy">This interactive demonstration probes the isolated kernel runtime, validates hypervisor and jail boundaries, audits network isolation across the private laboratory bridge, and measures process stability.</p>
  <div class="freebsdLab-IntroNotice">
    <div class="freebsdLab-IntroNoticeIcon">i</div>
    <div>
      <div class="freebsdLab-IntroNoticeLabel">Laboratory Diagnostics</div>
      <div class="freebsdLab-IntroNoticeText">All commands execute inside disposable, unprivileged runtimes connected via encrypted host-initiated SSH tunnels.</div>
    </div>
    <div class="freebsdLab-IntroNoticeMeta"><strong>FreeBSD</strong>demo + probe</div>
  </div>
</div>

## 1. Runtime Identity & Isolation Properties
Verify hypervisor guest status (`bhyve`), jail isolation, unprivileged process UID, and routing mutation privilege boundaries.

In [ ]:
import platform, sys, os, subprocess, socket

def run(cmd):
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return p.stdout.strip() if p.returncode == 0 else f'ERR ({p.returncode}): {p.stderr.strip()}'

print('=== RUNTIME IDENTITY & ISOLATION PROPERTIES ===')
print('Hypervisor (kern.vm_guest): ', run('sysctl -n kern.vm_guest 2>/dev/null || echo N/A'))
print('Jail ID (security.jail):    ', run('sysctl -n security.jail.jailed 2>/dev/null || echo 0'))
print('Platform:                   ', platform.platform())
print('Runtime Process:            ', f'UID={os.getuid()} (whoami={run("whoami")}), GID={os.getgid()}, PID={os.getpid()}')
print('Route Mutation Privilege:   ', run('route add default 172.31.254.1 2>&1'))
print('Passwordless Sudo:          ', run('sudo -n true 2>&1 || echo Denied / Password Required'))


## 2. Inbound Management Transport & Loopback Invariant
Audit guest listeners and confirm that Jupyter ZeroMQ channels bind strictly to `127.0.0.1`, carried over the host-initiated SSH tunnel.

In [ ]:
print('=== OPEN GUEST LISTENERS (SECURITY AUDIT) ===')
print(run('sockstat -4 -l 2>/dev/null || ss -tulpn 2>/dev/null || netstat -an'))
print()
print('=== ESTABLISHED TRANSPORT CONNECTIONS ===')
print(run('sockstat -4 -c 2>/dev/null || ss -tun 2>/dev/null || netstat -an'))


## 3. Subnet Neighbor & Private Bridge Port Isolation Scan
Probe neighboring addresses across the private laboratory subnet (`172.31.254.10`–`172.31.254.20`) to verify L2 private port isolation.

In [ ]:
import socket

def scan_ip_port(ip, port, timeout=0.25):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.settimeout(timeout)
    try:
        s.connect((ip, port))
        s.close()
        return 'OPEN'
    except socket.timeout:
        return 'FILTERED / TIMED OUT'
    except Exception as e:
        return f'REJECTED ({e})'

print('=== LOCAL SUBNET SCAN (172.31.254.10 - 172.31.254.20) ===')
ifconfig_out = run("ifconfig vtnet0 2>/dev/null | awk '/inet / {print $2}' || ifconfig vnet0 2>/dev/null | awk '/inet / {print $2}' || ip -4 addr show eth0 2>/dev/null | awk '/inet / {print $2}' | cut -d/ -f1")
my_ip = ifconfig_out if ifconfig_out else '172.31.254.10'

for host_num in range(10, 21):
    target = f'172.31.254.{host_num}'
    role = '(Self)' if target == my_ip else '(Neighbor VM)'
    status = scan_ip_port(target, 22)
    print(f'  {target:15s} {role:15s}: SSH/22={status}')

print()
print('=== HOST GATEWAY ACCESS PROBE (172.31.254.1) ===')
print('  Host Gateway SSH (22):   ', scan_ip_port('172.31.254.1', 22))
print('  Host Gateway HTTP (8888):', scan_ip_port('172.31.254.1', 8888))


## 4. Egress Failure Semantics & Routing Audit
Inspect the kernel routing table and demonstrate that external WAN sockets fail immediately at the local route level due to absence of a default gateway.

In [ ]:
print('=== ROUTING TABLE INSPECTION ===')
print(run('netstat -rn 2>/dev/null || ip route 2>/dev/null || route -n'))
print()
print('=== DIRECT OUTBOUND SOCKET ATTEMPTS ===')
for ip in ['1.1.1.1', '8.8.8.8']:
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.settimeout(1.0)
        s.connect((ip, 80))
        print(f'  {ip}:80 -> CONNECTED')
    except OSError as e:
        print(f'  {ip}:80 -> [Errno {e.errno}] {e.strerror}')
        print('         Mechanism: Local guest kernel rejection (no default route).')


## 5. Computation Durability & Memory Stability
Perform iterative array allocation and math operations to test CPU execution stability and memory persistence inside the runtime.

In [ ]:
import time

print('=== COMPUTATION & RUNTIME STABILITY TEST ===')
t0 = time.perf_counter()
data = [i ** 2 for i in range(1_000_000)]
total = sum(data)
elapsed = (time.perf_counter() - t0) * 1000
print(f'Allocated 1M ints, sum = {total}, completed in {elapsed:.2f}ms')
print('Memory allocation and execution pipeline healthy.')


## 6. Socket Metrics & File Descriptor Leak Check
Inspect the open file descriptors of the kernel process to verify clean socket lifecycle and prevent descriptor leakage.

In [ ]:
print('=== FILE DESCRIPTORS & PROCESS METRICS ===')
print('Current Process PID: ', os.getpid())
print('Open File Descriptors:')
print(run(f'procstat -f {os.getpid()} 2>/dev/null | head -n 15 || ls -la /proc/{os.getpid()}/fd 2>/dev/null | head -n 15'))
